# Projet 4 — Analyse exploratoire du dataset médical

**Objectif :** vérifier l'intégrité du CSV avant migration vers MongoDB, préparer le dictionnaire de données et produire une base de référence pour le contrôle après migration.

## 1. Chargement et structure générale

Importation des bibliothèques

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from bson.decimal128 import Decimal128
from pymongo import MongoClient
from collections import Counter
from bson.json_util import dumps
from datetime import datetime

Importation des données depuis le fichier `.csv`

In [ ]:
csv_path = Path("data/healthcare_dataset.csv")
if not csv_path.exists():
    raise FileNotFoundError("Placer healthcare_dataset.csv dans data/.")

df = pd.read_csv(csv_path)

print(f"Fichier : {csv_path.name}")
print(f"Lignes : {len(df):,} | Colonnes : {len(df.columns)}")
print(f"Taille : {csv_path.stat().st_size / 1_000_000:.2f} Mo")
display(pd.DataFrame({
    "colonne": df.columns,
    "type_pandas": df.dtypes.astype(str).to_numpy(),
    "valeurs_distinctes": df.nunique(dropna=False).to_numpy()
}))

### Choix de modélisation : les sous-documents MongoDB

Contrairement à une base de données relationnelle, MongoDB stocke les informations sous forme de documents BSON (format binaire proche du JSON). Ces documents peuvent contenir d'autres documents, appelés **sous-documents**.

Pour notre dataset médical, deux approches sont possibles :

- **Structure plate :** conserver les 15 colonnes du CSV au même niveau dans chaque document MongoDB.
- **Structure imbriquée :** regrouper les colonnes dans des sous-documents selon leur fonction.

**Nous avons choisi la structure avec sous-documents**, afin d'organiser les données de manière plus logique et plus lisible.

Chaque ligne du CSV représentera une hospitalisation et sera transformée en un document MongoDB contenant quatre sous-documents :

| Sous-document | Informations contenues |
|---|---|
| `patient` | Nom, âge, genre, groupe sanguin |
| `hospitalisation` | Dates d'admission et de sortie, médecin, hôpital, numéro de chambre, type d'admission |
| `medical` | Pathologie, médicament, résultats des examens |
| `facturation` | Assurance et montant facturé |

#### Avantages de cette organisation

- **Lisibilité :** les informations sont regroupées par thème.
- **Organisation :** les données relatives à une même hospitalisation restent réunies dans un seul document.
- **Évolutivité :** il est possible d'ajouter des champs dans un sous-document sans modifier tous les documents existants.
- **Exploitation de MongoDB :** cette organisation utilise les possibilités d'imbrication offertes par une base NoSQL documentaire.

#### Points de vigilance

Cette approche nécessite une transformation des données pendant la migration. Il faudra notamment vérifier que tous les champs du CSV sont correctement transférés et que leurs types sont adaptés à MongoDB.

De plus, le dataset ne possède pas d'identifiant unique de patient. Chaque document correspondra donc à **une hospitalisation et non à un patient unique**.

**Conclusion :** le choix des sous-documents permet d'obtenir une structure claire et adaptée aux données médicales, tout en facilitant leur organisation et leur évolution future.

## 2. Dictionnaire et types MongoDB envisagés

**Principe :** 1 ligne du fichier `.csv` = 1 document d'hospitalisation<br>
Les champs sont regroupés dans quatre sous-documents. La colonne `Name` ne permet **pas** de définir une identité patient unique.


In [ ]:
champs = [
    {"csv": "Name", "mongo": "patient.nom", "type_bson": "String"},
    {"csv": "Age", "mongo": "patient.age", "type_bson": "Int32"},
    {"csv": "Gender", "mongo": "patient.genre", "type_bson": "String"},
    {"csv": "Blood Type", "mongo": "patient.groupe_sanguin", "type_bson": "String"},

    {"csv": "Medical Condition", "mongo": "medical.pathologie", "type_bson": "String"},
    {"csv": "Medication", "mongo": "medical.medicament", "type_bson": "String"},
    {"csv": "Test Results", "mongo": "medical.resultat_test", "type_bson": "String"},

    {"csv": "Date of Admission", "mongo": "hospitalisation.date_admission", "type_bson": "Date"},
    {"csv": "Discharge Date", "mongo": "hospitalisation.date_sortie", "type_bson": "Date"},
    {"csv": "Doctor", "mongo": "hospitalisation.medecin", "type_bson": "String"},
    {"csv": "Hospital", "mongo": "hospitalisation.hopital", "type_bson": "String"},
    {"csv": "Room Number", "mongo": "hospitalisation.numero_chambre", "type_bson": "Int32"},
    {"csv": "Admission Type", "mongo": "hospitalisation.type_admission", "type_bson": "String"},

    {"csv": "Insurance Provider", "mongo": "facturation.assureur", "type_bson": "String"},
    {"csv": "Billing Amount", "mongo": "facturation.montant", "type_bson": "Double"}
]
dictionnaire = pd.DataFrame(champs)
print(f"Nombre de champs documentés : {len(dictionnaire)}")
print(f"Nombre de colonnes du CSV : {len(df.columns)}")
colonnes_manquantes = set(df.columns) - set(dictionnaire["csv"])
print("Colonnes non documentées :", colonnes_manquantes)
display(dictionnaire)

### Précisions de typage

- **Dates :** les chaînes `YYYY-MM-DD` sont converties en objets `datetime` UTC pour des dates MongoDB.
- **Montants :** `Decimal128` permet de préserver les valeurs décimales exactes du CSV. Les éventuels arrondis à deux décimales nécessitent une décision métier explicite.
- **Entiers :** `Age` et `Room Number` sont des nombres entiers.
- **Clé `_id` :** générer un identifiant technique stable depuis le fichier source et le numéro de ligne ; ne pas utiliser le nom, ni imposer l'unicité sur tout le document.

Le choix « un document par hospitalisation » évite d'inventer un identifiant unique de patient qui n'existe pas dans la source.

## 3. Intégrité des données : contrôles des valeurs nulles, vides et des doublons

Recherche des valeurs manquantes : (`.isna` renvoie true si valeur manquante) 

In [ ]:
print(df.isna().sum())
print("Total des valeurs manquantes :", df.isna().sum().sum())

`.isna` ne detecte pas les valeur vides remplis uniquement par des espaces.
Après avoir vérifié l'absence de valeurs manquantes, nous recherchons les chaînes de caractères vides ou contenant uniquement des espaces.

In [ ]:
valeurs_vides = df.replace(r"^\s*$", pd.NA, regex=True).isna().sum()
print(valeurs_vides)
print("Total des valeurs vides ou manquantes :", valeurs_vides.sum())

Recherche des doublons.<br>

In [ ]:
nb_doublons = df.duplicated().sum()

print("Nombre de doublons :", nb_doublons)

Résultat : 534 doublons ont été détectés parmi les 55 500 enregistrements, soit environ 0,96 % des données.<br>
Le fichier ne possédant pas d'identifiant unique d'hospitalisation, nous ne pouvons pas déterminer avec certitude si ces doublons proviennent d'erreurs ou correspondent à des enregistrements distincts.<br>
Suppression des doublons :

In [ ]:
df_clean = df.drop_duplicates().copy()

print("Lignes avant nettoyage :", len(df))
print("Lignes après nettoyage :", len(df_clean))
print("Lignes supprimées :", len(df) - len(df_clean))

## 4. Contrôles des dates et des durées d'hospitalisation

Vérification et conversion des dates.<br>
Les colonnes Date of Admission et Discharge Date sont initialement reconnues comme du texte par pandas.<br>
Nous utilisons pd.to_datetime() pour les convertir au format datetime, afin de préparer leur futur stockage dans MongoDB.<br>
Le paramètre errors="coerce" permet d'identifier les dates invalides en les remplaçant par NaT.<br>
Après conversion, nous vérifions l'absence de valeurs invalides dans les deux colonnes.<br>

In [ ]:
df_clean["Date of Admission"] = pd.to_datetime(
    df_clean["Date of Admission"], errors="coerce"
)
df_clean["Discharge Date"] = pd.to_datetime(
    df_clean["Discharge Date"], errors="coerce"
)
print(df_clean[["Date of Admission", "Discharge Date"]].dtypes)

In [ ]:
print(df_clean[["Date of Admission", "Discharge Date"]].isna().sum())

Vérification de la cohérence des dates (la date de sortie d'un patient n'est jamais antérieure à sa date d'admission).

In [ ]:
dates_incoherentes = df_clean[
    df_clean["Discharge Date"] < df_clean["Date of Admission"]
]
print("Nombre de dates incohérentes :", len(dates_incoherentes))

### Vérification des montants facturés.

Nous vérifions les valeurs de la colonne `Billing Amount` afin de détecter d'éventuelles anomalies avant la migration vers MongoDB.<br>
Les contrôles portent sur le type de données, la présence de montants négatifs ou nuls, ainsi que les valeurs minimales et maximales.<br>

**Résultats :**
- Les montants sont reconnus comme des nombres décimaux (`float64`).
- 106 montants négatifs sont présents après suppression des doublons.
- Aucun montant nul n'a été détecté.

Les montants négatifs peuvent correspondre à des remboursements, des avoirs ou des erreurs de saisie. **Leur origine ne pouvant pas être déterminée, nous choisissons de les conserver** afin de ne pas modifier les données sans justification.<br>
Lors de la migration vers MongoDB, nous prévoyons d'utiliser le type BSON `Decimal128` pour stocker les montants avec une précision décimale adaptée.

In [ ]:
print("Type :", df_clean["Billing Amount"].dtype)

print("Montants négatifs :", (df_clean["Billing Amount"] < 0).sum())
print("Montants nuls :", (df_clean["Billing Amount"] == 0).sum())

print("Montant minimum :", df_clean["Billing Amount"].min())
print("Montant maximum :", df_clean["Billing Amount"].max())

### Contrôle des données numériques

Nous vérifions la cohérence des colonnes `Age` et `Room Number` avant leur migration vers MongoDB.<br>
Les contrôles portent sur les types des données, leurs valeurs minimales et maximales, ainsi que la présence de valeurs potentiellement incohérentes.<br>

In [ ]:
print("Type :", df_clean["Age"].dtype)
print("Âge minimum :", df_clean["Age"].min())
print("Âge maximum :", df_clean["Age"].max())

print("Âges incohérents :", (
    (df_clean["Age"] < 0) | (df_clean["Age"] > 120)
).sum())

# Vérification des numéros de chambre

print("\nType :", df_clean["Room Number"].dtype)
print("Chambre minimum :", df_clean["Room Number"].min())
print("Chambre maximum :", df_clean["Room Number"].max())

print("Numéros de chambre invalides :", (
    df_clean["Room Number"] <= 0
).sum())

**Résultats :**

- Les deux colonnes sont reconnues comme des entiers (`int64`).
- Les âges sont compris entre 13 et 89 ans.
- Les numéros de chambre sont compris entre 101 et 500.
- Aucune valeur incohérente n'a été détectée selon les règles définies.

Ces deux colonnes ne nécessitent donc aucune correction à cette étape.<br>
Lors de la migration, elles pourront être stockées sous forme d'entiers BSON (`Int32`).

### Contrôle des variables catégorielles

Nous vérifions les différentes valeurs présentes dans les variables catégorielles du dataset à l'aide de la méthode `value_counts()` de pandas.

Cette vérification permet d'identifier d'éventuelles incohérences telles que des fautes de frappe, des différences de casse ou des valeurs inattendues.

Les contrôles portent sur le genre, le groupe sanguin, les pathologies, le type d'admission, les médicaments, les résultats des examens et les assurances.

In [ ]:
colonnes = [
    "Gender",
    "Blood Type",
    "Medical Condition",
    "Admission Type",
    "Medication",
    "Test Results",
    "Insurance Provider"
]

for colonne in colonnes:
    print(f"\n{colonne}")
    print(df_clean[colonne].value_counts())

**Résultat :** les catégories observées sont cohérentes et aucune anomalie de format évidente n'a été détectée.

Aucune correction n'est donc nécessaire sur ces colonnes avant leur migration vers MongoDB.

### Bilan du nettoyage des données

L'analyse du dataset a permis de vérifier la qualité et la cohérence des données avant leur migration vers MongoDB.

Le dataset initial contient **55 500 enregistrements répartis sur 15 colonnes**.

Les principales opérations réalisées sont :

- Suppression de 534 lignes strictement identiques.
- Conversion des dates au format `datetime`.
- Vérification des valeurs manquantes, des types de données et de la cohérence des informations.

Les 106 montants négatifs ont été conservés, leur origine ne pouvant pas être déterminée avec certitude.

Le DataFrame `df_clean` contient désormais **54 966 enregistrements**, sans doublons exacts ni valeurs manquantes.

Ce DataFrame constituera notre référence pour la migration vers MongoDB. Nous vérifierons après l'importation que le nombre d'enregistrements et les informations ont été correctement conservés.

### Transformation des données en documents MongoDB

Après le nettoyage des données, nous préparons leur transformation au format documentaire utilisé par MongoDB.

Nous choisissons de représenter chaque hospitalisation par un document contenant quatre sous-documents : `patient`, `medical`, `hospitalisation` et `facturation`.

Pour cela, nous récupérons une ligne du DataFrame avec `iloc[0]`, puis nous créons un dictionnaire Python dont les clés correspondent aux champs définis dans notre dictionnaire de données.

Cette organisation permet de regrouper les informations par thème tout en conservant les 15 champs du dataset original.

Nous effectuons également les premières adaptations des types de données, notamment pour les entiers et les dates.

In [ ]:
# on recupère une ligne
ligne = df_clean.iloc[0]

document = {
    "patient": {
        "nom": ligne["Name"],
        "age": int(ligne["Age"]),
        "genre": ligne["Gender"],
        "groupe_sanguin": ligne["Blood Type"]
    },

    "medical": {
        "pathologie": ligne["Medical Condition"],
        "medicament": ligne["Medication"],
        "resultat_test": ligne["Test Results"]
    },

    "hospitalisation": {
        "date_admission": ligne["Date of Admission"].to_pydatetime(),
        "date_sortie": ligne["Discharge Date"].to_pydatetime(),
        "medecin": ligne["Doctor"],
        "hopital": ligne["Hospital"],
        "numero_chambre": int(ligne["Room Number"]),
        "type_admission": ligne["Admission Type"]
    },

    "facturation": {
        "assureur": ligne["Insurance Provider"],
        "montant": ligne["Billing Amount"]
    }
}

print(document.keys())
print(document["patient"].keys())

**Résultat :** une ligne du CSV a été transformée en un dictionnaire Python structuré, prêt à être adapté aux types BSON puis inséré dans MongoDB.
Créer notre document.<br>
Nous allons utiliser un dictionnaire Python contenant quatre dictionnaires imbriqués.

### Automatisation de la transformation en documents MongoDB

Afin d'automatiser la transformation du dataset, nous avons créé une fonction `create_doc()`.<br>
Cette fonction reçoit une ligne du DataFrame et renvoie un dictionnaire Python contenant les quatre sous-documents définis précédemment.<br>
Nous utilisons ensuite une boucle `for` avec la méthode `iterrows()` pour parcourir les lignes de `df_clean` et appliquer la fonction à chacune d'elles.<br>
Les documents obtenus sont stockés dans une liste Python grâce à la méthode `append()`.<br>

In [ ]:
def create_doc(line):
    doc = {
        "patient": {
            "nom": line["Name"],
            "age": int(line["Age"]),
            "genre": line["Gender"],
            "groupe_sanguin": line["Blood Type"]
        },

        "medical": {
            "pathologie": line["Medical Condition"],
            "medicament": line["Medication"],
            "resultat_test": line["Test Results"]
        },

        "hospitalisation": {
            "date_admission": line["Date of Admission"].to_pydatetime(),
            "date_sortie": line["Discharge Date"].to_pydatetime(),
            "medecin": line["Doctor"],
            "hopital": line["Hospital"],
            "numero_chambre": int(line["Room Number"]),
            "type_admission": line["Admission Type"]
        },

        "facturation": {
            "assureur": line["Insurance Provider"],
            "montant": float(line["Billing Amount"])
        }
    }

    return doc

docs = []

for index, ligne in df_clean.iterrows():
    docs.append(create_doc(ligne))

print("Documents créés :", len(docs))

**Résultat :** les 54 966 lignes du dataset nettoyé ont été transformées en 54 966 documents structurés.<br>
Nous vérifions que le nombre de documents créés correspond au nombre de lignes du DataFrame afin de détecter une éventuelle perte d'enregistrements.

### Connexion à MongoDB

Nous utilisons la bibliothèque Python `pymongo` pour établir une connexion entre notre script et le serveur MongoDB installé localement.<br>
La classe `MongoClient` permet de communiquer avec le serveur à l'adresse `localhost`, sur le port `27017`.<br>
Nous exécutons ensuite la commande `ping` pour vérifier que le serveur MongoDB est accessible et répond correctement.<br>
Cette vérification permet de s'assurer que la connexion fonctionne avant de commencer la migration des données.<br>

In [ ]:
client = MongoClient("mongodb://localhost:27017/")
client.admin.command("ping")
print("Connexion à MongoDB réussie !")

### Création de la base MongoDB et test d'insertion

Nous sélectionnons une base de données nommée `healthcare` et une collection nommée `hospitalisations`.<br>
Avec MongoDB, la base de données et la collection sont créées automatiquement lors de la première insertion d'un document.<br>
Nous utilisons la méthode `insert_one()` de PyMongo pour insérer un premier document et vérifier le bon fonctionnement de la connexion et de l'écriture.<br>
MongoDB attribue automatiquement un identifiant unique `_id` au document.<br>
Nous vérifions ensuite la présence du document avec `count_documents()`.<br>
Après validation, nous supprimons le document de test afin de préparer la collection pour la migration complète.<br>

In [ ]:
# Sélection de la base de données
db = client["healthcare"]

# Sélection de la collection
collection = db["hospitalisations"]

# Insertion d'un document de test
result = collection.insert_one(docs[0].copy())
print("Document inséré avec l'identifiant :", result.inserted_id)
print("Documents dans la collection :", collection.count_documents({}))

# Suppression du document de test
collection.delete_one({"_id": result.inserted_id})
print("Documents restants :", collection.count_documents({}))

### Importation des données dans MongoDB

Après avoir préparé et structuré les données, nous procédons à leur insertion dans la collection `hospitalisations`.<br>
Nous utilisons la méthode `insert_many()` de PyMongo pour importer l'ensemble des documents.<br>
Avant l'importation, nous vérifions que la collection est vide afin d'éviter les insertions répétées.<br>
Une fois l'opération terminée, nous comparons le nombre de documents insérés avec le nombre d'enregistrements attendus.<br>
**Résultat attendu :** 54 966 documents importés dans MongoDB.<br>

In [ ]:
# Importation des documents dans MongoDB
if collection.count_documents({}) == 0:
    result = collection.insert_many(docs)

    print("Documents importés :", len(result.inserted_ids))
    print("Documents dans MongoDB :", collection.count_documents({}))
else:
    print("Importation annulée : la collection n'est pas vide.")

### Vérification de l'intégrité après migration
Après l'importation des données dans MongoDB, nous effectuons une vérification globale de la migration.

Les contrôles portent sur :
- Le nombre de documents importés.
- La présence des quatre sous-documents et des 15 champs.
- La conformité des principaux types de données.
- La comparaison des documents MongoDB avec ceux préparés en Python.

Cette dernière comparaison permet de vérifier qu'aucune donnée n'a été perdue, ajoutée ou modifiée pendant l'importation.

Si tous les contrôles retournent `True`, la migration est considérée comme validée pour ces critères.

In [ ]:
# Récupération des documents MongoDB
mongo_docs = list(collection.find({}, {"_id": 0}))

# Vérification du nombre
nb_ok = len(mongo_docs) == len(df_clean)

# Vérification de la structure (4 sous-documents, 15 champs)
structure_ok = all(
    set(doc) == {"patient", "medical", "hospitalisation", "facturation"}
    and sum(len(groupe) for groupe in doc.values()) == 15
    for doc in mongo_docs
)

# Vérification des types principaux
types_ok = structure_ok and all(
    isinstance(doc["patient"]["age"], int)
    and isinstance(doc["hospitalisation"]["date_admission"], datetime)
    and isinstance(doc["hospitalisation"]["date_sortie"], datetime)
    and isinstance(doc["facturation"]["montant"], float)
    for doc in mongo_docs
)

# Comparaison des données avant et après insertion
def signature(doc):
    return dumps({k: v for k, v in doc.items() if k != "_id"}, sort_keys=True)

donnees_ok = (
    Counter(signature(doc) for doc in docs)
    == Counter(signature(doc) for doc in mongo_docs)
)

print("Nombre de documents conforme :", nb_ok)
print("Structure conforme :", structure_ok)
print("Types conformes :", types_ok)
print("Données identiques :", donnees_ok)

print("\nMigration validée :", all([nb_ok, structure_ok, types_ok, donnees_ok]))

### Création des index MongoDB

Afin d'optimiser les performances des recherches, nous créons trois index sur les champs susceptibles d'être fréquemment utilisés : la pathologie, la date d'admission et l'hôpital.

Ces index permettent à MongoDB de retrouver plus efficacement les documents correspondant à certains critères, sans nécessairement parcourir toute la collection.

Nous utilisons la méthode `create_index()` pour créer les index, puis `list_indexes()` pour vérifier leur présence.

Leur efficacité dépendra des requêtes exécutées et pourra être évaluée avec la méthode `explain()`.

Les index sont créés sans contrainte d'unicité, car plusieurs hospitalisations peuvent partager une même pathologie, une même date ou un même hôpital.

In [ ]:
# Création des index
collection.create_index("medical.pathologie")
collection.create_index("hospitalisation.date_admission")
collection.create_index("hospitalisation.hopital")

# Vérification des index
print("Index disponibles :")
for index in collection.list_indexes():
    print(index["name"])

### Tests des opérations CRUD

Afin de vérifier le fonctionnement des principales opérations MongoDB, nous réalisons des tests CRUD à l'aide de la bibliothèque PyMongo.

Nous utilisons une collection `tests_crud` distincte de notre collection `hospitalisations`, afin de ne pas modifier les données médicales importées.

Les quatre opérations testées sont :

- **Create :** insertion d'un document avec `insert_one()`.
- **Read :** récupération du document avec `find_one()`.
- **Update :** modification d'un champ avec `update_one()` et l'opérateur `$set`.
- **Delete :** suppression du document avec `delete_one()`.

Ces tests permettent de vérifier que les opérations de création, de lecture, de modification et de suppression fonctionnent correctement.

Le document créé pour les tests est supprimé à la fin de l'exécution.

In [ ]:
# Collection dédiée aux tests
test_collection = db["tests_crud"]

# CREATE : ajouter un document
result = test_collection.insert_one({
    "nom": "Test",
    "statut": "Nouveau"
})

id_test = result.inserted_id
print("CREATE :", id_test)

# READ : lire le document
doc = test_collection.find_one({"_id": id_test})
print("READ :", doc["statut"])

# UPDATE : modifier le document
test_collection.update_one(
    {"_id": id_test},
    {"$set": {"statut": "Modifié"}}
)
print("UPDATE :", test_collection.find_one({"_id": id_test})["statut"])

# DELETE : supprimer le document
test_collection.delete_one({"_id": id_test})
print("DELETE :", test_collection.find_one({"_id": id_test}))

In [ ]:
Test connection mongo

In [23]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

# Chargement des identifiants
load_dotenv(".env")

# Connexion à MongoDB Docker
client_docker = MongoClient(
    "mongodb://127.0.0.1:27018/",
    username=os.getenv("MONGO_ROOT_USER"),
    password=os.getenv("MONGO_ROOT_PASSWORD"),
    authSource="admin"
)

# Vérification
client_docker.admin.command("ping")

print("Connexion à MongoDB Docker réussie !")

client_docker.close()

Connexion à MongoDB Docker réussie !
